# Exploratory Data Analysis — Emergency Sound Detection

Run this after populating `data/raw/<class>/*.wav` (see `src/download_data.py`).

Goals:
1. Check class balance
2. Inspect clip durations
3. Listen to sample waveforms + spectrograms per class

In [ ]:
import os
import sys
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

from config import RAW_DATA_DIR, CLASSES, SAMPLE_RATE

## 1. Class balance

In [ ]:
counts = {}
for c in CLASSES:
    d = os.path.join(RAW_DATA_DIR, c)
    counts[c] = len(os.listdir(d)) if os.path.isdir(d) else 0

plt.figure(figsize=(8, 4))
plt.bar(counts.keys(), counts.values())
plt.title('Class distribution')
plt.ylabel('# clips')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
print(counts)

## 2. Clip durations

In [ ]:
durations = []
for c in CLASSES:
    d = os.path.join(RAW_DATA_DIR, c)
    if not os.path.isdir(d):
        continue
    for f in os.listdir(d)[:50]:  # sample first 50 per class to keep this fast
        try:
            dur = librosa.get_duration(path=os.path.join(d, f))
            durations.append(dur)
        except Exception:
            pass

plt.figure(figsize=(8, 4))
plt.hist(durations, bins=30)
plt.title('Clip duration distribution (sampled)')
plt.xlabel('seconds')
plt.show()

## 3. Waveform + spectrogram per class

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 2, figsize=(12, 3 * len(CLASSES)))

for i, c in enumerate(CLASSES):
    d = os.path.join(RAW_DATA_DIR, c)
    if not os.path.isdir(d) or not os.listdir(d):
        continue
    fpath = os.path.join(d, os.listdir(d)[0])
    y, sr = librosa.load(fpath, sr=SAMPLE_RATE)

    axes[i, 0].plot(y)
    axes[i, 0].set_title(f'{c} — waveform')

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(log_mel, sr=sr, ax=axes[i, 1])
    axes[i, 1].set_title(f'{c} — mel-spectrogram')

plt.tight_layout()
plt.show()